# Working-day seasonality

Uses `expiry_dates.csv` plus `market_data.db`. Prices are joined only on real common settlement dates, so missing exchange/off days stay as blanks in the exported Excel.


In [13]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

from analysis.plotting import SeasonalityPlotter

ROOT = Path.cwd()
DB_PATH = ROOT / "market_data.db"
EXPIRY_PATH = ROOT / "expiry_dates.csv"
WINDOW_WORKING_DAYS = 300

SYMBOL = "CO"
EXPRESSION = "X26-Z26"
START_YEAR = 2020
END_YEAR = 2026

db = sqlite3.connect(f"file:{DB_PATH.resolve().as_posix()}?mode=ro", uri=True)
expiry_dates = pd.read_csv(EXPIRY_PATH, parse_dates=["expiry_date"])
expiry_lookup = expiry_dates.set_index(["symbol", "contract_code"])["expiry_date"]


In [14]:
def get_history(symbol, contract_code):
    history = pd.read_sql_query(
        """
        SELECT trading_date AS date, price
        FROM seac_settlements
        WHERE symbol = ? AND contract_code = ?
        ORDER BY trading_date
        """,
        db,
        params=(symbol, contract_code),
        parse_dates=["date"],
    )
    return history.set_index("date")


def get_expiry(symbol, contract_code, history):
    return expiry_lookup.get((symbol, contract_code), history.index.max())


def parse_expression(expression):
    terms = []
    for term in expression.replace(" ", "").replace("-", "+-").split("+"):
        if not term:
            continue
        if "*" in term:
            coefficient, code = term.split("*")
            terms.append((float(coefficient), code))
        else:
            terms.append((-1.0, term[1:]) if term.startswith("-") else (1.0, term))
    return terms


In [15]:
def expression_season(symbol, expression, season_year, window=WINDOW_WORKING_DAYS):
    terms = parse_expression(expression)
    reference_year = int(terms[0][1][1:])
    legs = []
    anchor_code = None
    anchor_history = None

    for coefficient, code in terms:
        mapped_code = f"{code[0]}{(season_year + int(code[1:]) - reference_year) % 100:02d}"
        history = get_history(symbol, mapped_code)
        if history.empty:
            return None
        legs.append((history["price"] * coefficient).rename(mapped_code))
        if anchor_code is None:
            anchor_code, anchor_history = mapped_code, history

    values = pd.concat(legs, axis=1, join="inner").dropna().sum(axis=1)
    expiry = get_expiry(symbol, anchor_code, anchor_history)
    output = pd.DataFrame({"value": values, "date": values.index})
    output["working_days_to_expiry"] = -np.busday_count(
        output.index.values.astype("datetime64[D]"),
        np.datetime64(expiry.date()),
    )
    output = output[output["working_days_to_expiry"].between(-window, 0)]
    return output.set_index("working_days_to_expiry")


def expression_seasonality(symbol, expression, start_year, end_year, window=WINDOW_WORKING_DAYS):
    seasons = {}
    for year in range(start_year, end_year + 1):
        season = expression_season(symbol, expression, year, window)
        if season is not None and not season.empty:
            seasons[year] = season

    raw_combined = pd.concat({year: season["value"] for year, season in seasons.items()}, axis=1).sort_index()
    combined = raw_combined.interpolate(method="linear", limit_area="inside")
    tickvals = [-400, -300, -200, -100, 0]
    current_year = max(seasons)
    terms = parse_expression(expression)
    reference_year = int(terms[0][1][1:])
    anchor_code = f"{terms[0][1][0]}{(current_year + int(terms[0][1][1:]) - reference_year) % 100:02d}"
    anchor_history = get_history(symbol, anchor_code)
    anchor_expiry = get_expiry(symbol, anchor_code, anchor_history)
    ticktext = []
    for tick in tickvals:
        date = pd.Timestamp(np.busday_offset(np.datetime64(anchor_expiry.date()), tick, roll="backward"))
        label = "Expiry" if tick == 0 else str(tick)
        ticktext.append(f"{label}<br>{date.strftime('%Y-%m-%d')}")
    ticks = (tickvals, ticktext)
    return {
        "series": seasons,
        "raw_combined": raw_combined,
        "combined": combined,
        "average": combined.mean(axis=1),
        "ticks": ticks,
        "xaxis_title": "Working days to expiry",
    }


In [16]:
result = expression_seasonality(SYMBOL, EXPRESSION, START_YEAR, END_YEAR)

coverage = pd.DataFrame({
    year: {
        "observations": len(series),
        "first_date": series["date"].min(),
        "last_date": series["date"].max(),
        "first_offset": series.index.min(),
        "last_offset": series.index.max(),
    }
    for year, series in result["series"].items()
}).T

print("Years:", list(result["series"].keys()))
print("Raw NA count:", int(result["raw_combined"].isna().sum().sum()))
print("Interpolated NA count:", int(result["combined"].isna().sum().sum()))
display(coverage)
display(result["combined"].tail(15))


Years: [2020, 2021, 2022, 2023, 2024, 2025, 2026]
Raw NA count: 65
Interpolated NA count: 46


,observations,first_date,last_date,first_offset,last_offset
2020,298,2019-08-07 00:00:00,2020-09-30 00:00:00,-300,0
2021,298,2020-08-06 00:00:00,2021-09-30 00:00:00,-300,0
2022,300,2021-08-06 00:00:00,2022-09-30 00:00:00,-300,0
2023,298,2022-08-05 00:00:00,2023-09-29 00:00:00,-300,0
2024,298,2023-08-07 00:00:00,2024-09-30 00:00:00,-300,0
2025,298,2024-08-06 00:00:00,2025-09-30 00:00:00,-300,0
2026,252,2025-08-06 00:00:00,2026-07-28 00:00:00,-300,-46


,2020,2021,2022,2023,2024,2025,2026
working_days_to_expiry,,,,,,,
-14,-0.57,0.66,1.07,0.60,0.36,0.33,NaN
-13,-0.60,0.61,0.95,0.73,0.44,0.36,NaN
-12,-0.67,0.63,0.98,0.68,0.54,0.45,NaN
-11,-0.62,0.68,1.23,0.76,0.56,0.40,NaN
-10,-0.57,0.69,1.30,0.89,0.60,0.46,NaN
-9,-0.52,0.78,1.34,1.14,0.69,0.49,NaN
-8,-0.53,0.85,1.22,1.20,0.77,0.52,NaN
-7,-0.52,0.79,1.03,1.18,0.87,0.64,NaN
-6,-0.47,0.80,0.93,1.04,0.80,0.60,NaN


In [17]:
SeasonalityPlotter().plot_seasonality(
    result,
    title=f"{SYMBOL} {EXPRESSION} - working-day seasonality",
)


In [12]:
OUTPUT_FILE = ROOT / f"{SYMBOL}_{EXPRESSION.replace(' ', '').replace('*', 'x')}_{START_YEAR}_{END_YEAR}_working_days.xlsx"

with pd.ExcelWriter(OUTPUT_FILE) as writer:
    result["combined"].to_excel(writer, sheet_name="combined_interpolated")
    result["raw_combined"].to_excel(writer, sheet_name="combined_raw")
    result["average"].rename("average").to_excel(writer, sheet_name="average")
    coverage.to_excel(writer, sheet_name="coverage")

print("Saved:", OUTPUT_FILE.name)
print("Raw NA count:", int(result["raw_combined"].isna().sum().sum()))
print("Interpolated NA count:", int(result["combined"].isna().sum().sum()))


Saved: CO_X26-Z26_2020_2026_working_days.xlsx
Raw NA count: 72
Interpolated NA count: 46
